In [ ]:
# Phase 1B: Optimized & Resumable Archive Enrichment (Strict India)
import pandas as pd
import numpy as np
import requests
import time
import os
import sys
import reverse_geocoder as rg
from global_land_mask import globe
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- CONFIGURATION ---
INPUT_ARCHIVE = 'data/raw/filtered_2024_2026.csv'
OUTPUT_UNIFIED = 'data/raw/unified_archive_data.csv'
HISTORICAL_API = "https://archive-api.open-meteo.com/v1/archive"
SAMPLE_SIZE = 50000  # Managed size for API limits and training efficiency
BATCH_SIZE = 50      # Locations per API Call (Safe limit)

# Setup Session with Retries
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504, 429])
session.mount('https://', HTTPAdapter(max_retries=retries))

# ---------------------------------------------------------
# 1. LOAD & SAMPLE FIRE DATA
# ---------------------------------------------------------
print(f"📂 Loading Archive: {INPUT_ARCHIVE}...")
try:
    df_raw = pd.read_csv(INPUT_ARCHIVE)
    df_raw.columns = [c.lower() for c in df_raw.columns]
    
    # Rough Box Filter for faster processing
    df_raw = df_raw[(df_raw['latitude'] >= 6) & (df_raw['latitude'] <= 38) & 
                    (df_raw['longitude'] >= 68) & (df_raw['longitude'] <= 98)].copy()

    if len(df_raw) > SAMPLE_SIZE:
        print(f"✂️ Sampling {SAMPLE_SIZE} points for processing...")
        df_fire = df_raw.sample(n=SAMPLE_SIZE, random_state=42).copy()
    else:
        df_fire = df_raw.copy()

    # Strict Border Check for Fire Points
    print("🧐 Verifying India borders for fire points...")
    coords = list(zip(df_fire['latitude'], df_fire['longitude']))
    df_fire['country'] = [x['cc'] for x in rg.search(coords)]
    df_india_fire = df_fire[df_fire['country'] == 'IN'].copy()
    
    # Ensure standard columns
    if 'acq_time' not in df_india_fire.columns:
        df_india_fire['acq_time'] = 1200
    
    df_india_fire = df_india_fire[['latitude', 'longitude', 'acq_date', 'acq_time']].copy()
    df_india_fire['fire_detected'] = 1
    print(f"✅ Fire points verified: {len(df_india_fire)}")

except FileNotFoundError:
    print(f"❌ Error: {INPUT_ARCHIVE} not found.")
    sys.exit()

# ---------------------------------------------------------
# 2. GENERATE SAFE POINTS (STRICT INDIA & LAND)
# ---------------------------------------------------------
target_count = len(df_india_fire)
print(f"⚖️ Generating {target_count} Safe Points inside India...")
safe_points = []
possible_dates = df_india_fire['acq_date'].unique()

while len(safe_points) < target_count:
    needed = target_count - len(safe_points)
    lats = np.random.uniform(6.0, 38.0, needed * 3)
    lons = np.random.uniform(68.0, 98.0, needed * 3)
    
    # Filter 1: Land Check
    is_land = globe.is_land(lats, lons)
    lats, lons = lats[is_land], lons[is_land]
    
    if len(lats) > 0:
        # Filter 2: Border Check
        geo_results = rg.search(list(zip(lats, lons)))
        for i, res in enumerate(geo_results):
            if res['cc'] == 'IN':
                safe_points.append({
                    'latitude': lats[i],
                    'longitude': lons[i],
                    'acq_date': np.random.choice(possible_dates),
                    'acq_time': 1200,
                    'fire_detected': 0
                })
                if len(safe_points) >= target_count: break
    print(f"   > Progress: {len(safe_points)}/{target_count} safe points...", end='\r')

df_safe = pd.DataFrame(safe_points)

# Define Master DataFrame for Enrichment
master_df = pd.concat([df_india_fire, df_safe], ignore_index=True)
master_df = master_df.sample(frac=1).reset_index(drop=True)
print(f"\n✅ Balanced dataset ready with {len(master_df)} rows.")

# ---------------------------------------------------------
# 3. RESUMABLE ENRICHMENT ENGINE
# ---------------------------------------------------------
def fetch_with_backoff(params):
    for attempt in range(5):
        try:
            r = session.get(HISTORICAL_API, params=params, timeout=30)
            if r.status_code == 429:
                wait = (attempt + 1) * 30 
                print(f"🛑 Rate Limited. Cooling down for {wait}s...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            time.sleep(5)
    return None

# Check Existing Progress
if os.path.exists(OUTPUT_UNIFIED):
    df_existing = pd.read_csv(OUTPUT_UNIFIED)
    # Fix: Convert both components to string for safe concatenation
    finished_keys = set(df_existing['latitude'].astype(str) + df_existing['acq_date'].astype(str))
    print(f"🔄 Resuming... {len(finished_keys)} rows already finished.")
else:
    finished_keys = set()
    master_df.head(0).to_csv(OUTPUT_UNIFIED, index=False)

# Start API Grouping by Date
master_df['acq_date'] = pd.to_datetime(master_df['acq_date']).dt.strftime('%Y-%m-%d')
grouped = master_df.groupby('acq_date')

print(f"🚀 Starting weather enrichment...")
for date_str, group in grouped:
    # Filter group to only process new rows
    group_to_process = group[~(group['latitude'].astype(str) + date_str).isin(finished_keys)]
    
    if group_to_process.empty:
        continue

    for j in range(0, len(group_to_process), BATCH_SIZE):
        batch = group_to_process.iloc[j : j + BATCH_SIZE]
        params = {
            "latitude": ",".join(map(str, batch['latitude'])),
            "longitude": ",".join(map(str, batch['longitude'])),
            "start_date": date_str, "end_date": date_str,
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m"
        }
        
        data = fetch_with_backoff(params)
        if data:
            results_list = data if isinstance(data, list) else [data]
            enriched_batch = []
            for idx, (row_idx, row) in enumerate(batch.iterrows()):
                try:
                    res = results_list[idx]
                    if 'hourly' in res:
                        # Extract data for 12:00 PM (Noon)
                        enriched_batch.append({
                            **row.to_dict(),
                            'temperature_2m': res['hourly']['temperature_2m'][12],
                            'relative_humidity_2m': res['hourly']['relative_humidity_2m'][12],
                            'wind_speed_10m': res['hourly']['wind_speed_10m'][12]
                        })
                except IndexError:
                    continue
            
            if enriched_batch:
                pd.DataFrame(enriched_batch).to_csv(OUTPUT_UNIFIED, mode='a', header=False, index=False)
        
        print(f"   > Processed Date: {date_str} | Batch: {j//BATCH_SIZE + 1}", end='\r')
        time.sleep(2) # Baseline polite delay

print("\n✅ Enrichment Complete. Data saved to 'data/raw/unified_archive_data.csv'.")